In [ ]:
# Install required packages
# Run this cell once before executing the rest of the notebook.
%pip install numpy pandas scikit-learn matplotlib seaborn torch torchvision Pillow joblib tqdm python-dotenv google-auth google-auth-oauthlib google-api-python-client scikit-image scipy


# Heat Treatment Parameter Prediction — `database-debug` worksheet

This notebook predicts **heat treatment parameters** (Holding Temperature, Holding Time) from chemical composition and CNN image features using the SEM Microstructure Heat Treatment Prediction pipeline.

**Dataset**: `metadata_debug.csv` — exported from the `database-debug` worksheet of the project Google Sheet. Same schema as `metadata_latest.csv`; the only deliberate difference from `microstructure_demo.ipynb` is the data source (and the run scope, which is `debug` so artefacts don't overwrite the demo run family).

**Targets**: Cycle 1 HoldingTemp + HoldingTime (primary), with Cycle 2 extension

**Approach**: Multi-output ensemble regression (RF, GBR, AdaBoost)

**Sections**:
1. **Data Loading & Exploration** - Load dataset, analyze target availability across cycles
2. **Target Extraction** - Define targets, handle missing data, prevent leakage
3. **Tabular Feature Preprocessing** - Chemical composition features
4. **Image Feature Extraction** - Simulated CNN backbone features
5. **Model Training & Evaluation** - Multi-output ensemble models
6. **Visualization** - Per-target predictions and residual analysis
7. **Extension** - Cycle 2 multi-output prediction

In [ ]:
# Standard imports
import os
import re
import sys
import time
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path for local imports
sys.path.insert(0, os.path.abspath('..'))

# Project imports
from src import hyperparams as hp
from src.config import Config, PreprocessingConfig, MissingDataConfig, ScalingConfig, EncodingConfig
from src.preprocessing import FeaturePreprocessor
from src.extraction import FeatureExtractor, BackboneRegistry, MorphologicalExtractor, MorphologyConfig
from src.extraction.extractor import ExtractionConfig
from src.model_trainer import (
    ModelTrainer,
    build_ensemble_models,
    evaluate_model,
    plot_predictions,
    plot_learning_curves,
    plot_model_comparison
)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Display settings
pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

from src.features import FeaturePipeline

from src.run_store import RunStore
from src import metrics_viz


## 1. Data Loading & Exploration

The `metadata_debug.csv` dataset is the export of the `database-debug` worksheet and contains alloy samples with:
- **Chemical Composition**: C, Mn, Si, Cr, P, S, Mo, Cu, Ni, Al, Nb, V, Ti, Fe
- **Heat Treatment Parameters**: Up to 4 cycles, each with HoldingTemp, HoldingTime, CRate
- **Metadata**: Alloy names, sample IDs, image references

The CSV has a category header row, so we load with `header=1` and filter out empty padding rows.

In [ ]:
# ── RunStore: allocate this run's output directory ──────────────────────────
_store = RunStore("debug")
_run_dir, _run_id = _store.start()
print(f"Run directory: {_run_dir}")

# Load the metadata dataset
from src.column_sanitizer import sanitize_dataframe, sanitize_column

df_raw = pd.read_csv("../datasets/metadata_debug.csv", header=1)
print(f"Raw dataset shape: {df_raw.shape}")

# Sanitize all column names to [a-z0-9_] immediately after loading.
# All downstream code uses sanitized names exclusively.
df_raw = sanitize_dataframe(df_raw)

# Filter to rows with actual data (non-null chemical composition)
df = df_raw[df_raw["c"].notna()].copy().reset_index(drop=True)
print(f"Usable samples: {df.shape[0]} (filtered from {len(df_raw)} raw rows)")
print(f"Columns: {len(df.columns)}")
df.head()


In [ ]:
# Display all column names
print("=" * 60)
print("DATASET COLUMNS")
print("=" * 60)

for i, col in enumerate(df.columns):
    print(f"{i:3d}. {col}")

# Identify target-related columns and their availability
print("\n" + "=" * 60)
print("TARGET COLUMN AVAILABILITY (HoldingTemp, HoldingTime, CRate)")
print("=" * 60)

for col in df.columns:
    if any(kw in col for kw in ['holdingtemp', 'holdingtime', 'crate']):
        nn = df[col].notna().sum()
        print(f"  {repr(col):55s}  {nn:3d}/{len(df)} ({nn/len(df)*100:.0f}%)")

In [ ]:
# Examine data types and missing values
print("Data Types and Missing Values:")
print("-" * 60)

info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isna().sum(),
    'null_pct': (df.isna().sum() / len(df) * 100).round(1)
})

# Show columns with significant data
info_df[info_df['non_null'] > 0].head(30)

In [ ]:
# Target availability heatmap by cycle
# Column names are sanitized: e.g. cycle1_holdingtemp_degc, cycle1_holdingtime_min, cycle1_crate_degc_s
target_params = ['holdingtemp', 'holdingtime', 'crate']
target_labels = ['HoldingTemp', 'HoldingTime', 'CRate']
cycles = ['cycle1', 'cycle2', 'cycle3', 'cycle4']
cycle_labels = ['Cycle1', 'Cycle2', 'Cycle3', 'Cycle4']

availability = pd.DataFrame(index=cycle_labels, columns=target_labels, dtype=float)

for col in df.columns:
    for cycle, clabel in zip(cycles, cycle_labels):
        for param, plabel in zip(target_params, target_labels):
            if col.startswith(cycle + '_') and param in col:
                availability.loc[clabel, plabel] = df[col].notna().sum() / len(df) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap
sns.heatmap(availability.astype(float), annot=True, fmt='.0f', cmap='RdYlGn',
            vmin=0, vmax=100, ax=axes[0], cbar_kws={'label': '% Available'})
axes[0].set_title('Target Availability by Cycle (%)')
axes[0].set_ylabel('Cycle')

# Missing data bar chart for first 30 columns
missing_pct = (df.iloc[:, :30].isna().sum() / len(df) * 100)
colors = ['#2ecc71' if x < 50 else '#e74c3c' for x in missing_pct]
axes[1].bar(range(len(missing_pct)), missing_pct, color=colors)
axes[1].set_xlabel('Column Index')
axes[1].set_ylabel('Missing %')
axes[1].set_title('Missing Data by Column (First 30)')
axes[1].axhline(y=50, color='orange', linestyle='--', label='50% threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Compute conclusions dynamically from actual data
c1_temp_pct = availability.loc['Cycle1', 'HoldingTemp']
c1_time_pct = availability.loc['Cycle1', 'HoldingTime']
c2_temp_pct = availability.loc['Cycle2', 'HoldingTemp']
c2_time_pct = availability.loc['Cycle2', 'HoldingTime']
crate_max   = availability['CRate'].max()
c34_max     = availability.loc[['Cycle3', 'Cycle4'], ['HoldingTemp', 'HoldingTime']].max().max()

print("\nConclusion (computed from data):")
print(f"  - Cycle 1 HoldingTemp & HoldingTime: {c1_temp_pct:.0f}% / {c1_time_pct:.0f}% available -> PRIMARY TARGETS")
print(f"  - Cycle 2 HoldingTemp & HoldingTime: {c2_temp_pct:.0f}% / {c2_time_pct:.0f}% available -> EXTENSION TARGETS")
print(f"  - CRate: {crate_max:.0f}% max available across all cycles -> EXCLUDED")
print(f"  - Cycles 3-4: {c34_max:.0f}% max available -> EXCLUDED")
print()
print("Note: target columns are NOT imputed — these percentages reflect")
print("      actual label availability. Missing targets = excluded samples.")

In [ ]:
# Examine unique alloy types
print("Alloy Types in Dataset:")
print("-" * 40)
alloy_counts = df['alloy'].value_counts()
print(alloy_counts)

# Visualize alloy distribution
fig, ax = plt.subplots(figsize=(10, 5))
alloy_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_ylabel('Alloy Type')
ax.set_title(f'Distribution of Alloy Types ({len(df)} samples)')
plt.tight_layout()
plt.show()

## 2. Target Extraction

**Primary targets**: Cycle 1 HoldingTemp (C) and HoldingTime (min) — 82/88 samples available

**Strategy**:
- CRate excluded due to extreme sparsity (9% for Cycle 1, 0% for Cycles 2-4)
- Cycles 3-4 excluded (<20% availability)
- Samples missing Cycle 1 targets are dropped from training
- All heat treatment columns excluded from input features to prevent target leakage

In [ ]:
# Target columns — use sanitized names directly (no fuzzy matching needed)
# since column names are normalized at load time.
def find_column(df, sanitized_name):
    """Return sanitized_name if present in df.columns, else None."""
    return sanitized_name if sanitized_name in df.columns else None

TARGET_PATTERNS = {
    "Cycle1_HoldingTemp": "cycle1_holdingtemp_degc",
    "Cycle1_HoldingTime": "cycle1_holdingtime_min",
    "Cycle2_HoldingTemp": "cycle2_holdingtemp_degc",
    "Cycle2_HoldingTime": "cycle2_holdingtime_min",
}

target_col_map = {}
print("Target column mapping:")
for key, col in TARGET_PATTERNS.items():
    actual = find_column(df, col)
    if actual:
        target_col_map[key] = actual
        nn = df[actual].notna().sum()
        print(f"  {key:<25} -> {repr(actual)}  ({nn}/{len(df)} available)")
    else:
        print(f"  {key:<25} -> NOT FOUND")


In [ ]:
# Build Cycle 1 working subset
c1_temp_col = target_col_map["Cycle1_HoldingTemp"]
c1_time_col = target_col_map["Cycle1_HoldingTime"]

c1_mask = df[c1_temp_col].notna() & df[c1_time_col].notna()
df_c1 = df[c1_mask].copy().reset_index(drop=True)

target_columns = [c1_temp_col, c1_time_col]
Y_c1 = df_c1[target_columns].values.astype(np.float64)

print(f"Cycle 1 subset: {len(df_c1)} samples")
print(f"Target columns: {target_columns}")
for i, col in enumerate(target_columns):
    vals = Y_c1[:, i]
    print(f"  {col}: range=[{vals.min():.1f}, {vals.max():.1f}], mean={vals.mean():.1f}, std={vals.std():.1f}")


## 3. Tabular Feature Preprocessing

Input features are **chemical composition only** — all heat treatment parameters are excluded to prevent target leakage. In `metadata_debug.csv`, all chemical columns are `float64` (no text-in-numeric issues).

In [ ]:
# Input features: all columns EXCEPT HoldingTemp/HoldingTime across all cycles.
# Using sanitized column names throughout.
TARGET_COLUMNS_TO_EXCLUDE = [
    "cycle1_holdingtemp_degc", "cycle1_holdingtime_min",
    "cycle2_holdingtemp_degc", "cycle2_holdingtime_min",
    "cycle3_holdingtemp_degc", "cycle3_holdingtime_min",
    "cycle4_holdingtemp_degc", "cycle4_holdingtime_min",
]

feature_columns = [
    c for c in df_c1.columns
    if c not in TARGET_COLUMNS_TO_EXCLUDE
]

print(f"Selected {len(feature_columns)} input features (all columns minus HoldingTemp/HoldingTime):")
for col in feature_columns:
    nn = df_c1[col].notna().sum()
    print(f"  - {col:35s}  {nn:3d}/{len(df_c1)} ({nn/len(df_c1)*100:.0f}%) non-null  dtype={str(df_c1[col].dtype):10s}")


In [ ]:
# Create preprocessing configuration
preproc_config = PreprocessingConfig(
    missing_data=MissingDataConfig(
        column_drop_threshold=0.95,
        row_fill_threshold=0.50,
        numeric_fill_strategy="median",
        categorical_fill_strategy="mode",
        mice_max_iter=10,
    ),
    scaling=ScalingConfig(method="standard", enabled=True),
    encoding=EncodingConfig(categorical="onehot", max_categories=50),
)

MICE_COLUMNS      = ["cr", "mo", "s", "p", "ni", "al"]  # p: 52% missing, benefits from MICE
INDICATOR_COLUMNS = ["ti", "nb", "v"]

print("Preprocessing Configuration:")
print(f"  Column drop threshold: {preproc_config.missing_data.column_drop_threshold}")
print(f"  Row fill threshold:    {preproc_config.missing_data.row_fill_threshold}")
print(f"  Numeric fill strategy: {preproc_config.missing_data.numeric_fill_strategy}")
print(f"  Scaling method:        {preproc_config.scaling.method}")
print(f"  MICE columns:          {MICE_COLUMNS}")
print(f"  Indicator columns:     {INDICATOR_COLUMNS}")
print(f"  Note: cu (90.6% missing) uses median — too sparse for MICE")
print(f"  Indicator columns:     {INDICATOR_COLUMNS}")

In [ ]:
# Explicit column type overrides for columns whose pandas dtype is ambiguous.
COLUMN_TYPE_OVERRIDES = {
    "num_cycles":             "categorical",
    "cycle1_crate_degc_s":    "categorical",
    "cycle1_rtemp":           "categorical",
    "cycle1_qtemp":           "categorical",
    "cycle2_rtemp_degc":      "categorical",
    "cycle3_rtemp":           "categorical",
    "cycle3_qtemp":           "categorical",
    "cycle4_qtemp":           "categorical",
    "heat_treatment_type":    "categorical",
    "id":                     "unique_string",
}

# Split BEFORE fitting the preprocessor to avoid imputer/MICE leakage.
from sklearn.model_selection import train_test_split

idx = np.arange(len(df_c1))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=42)
idx_train, idx_val   = train_test_split(idx_trainval, test_size=0.15, random_state=42)

df_train_c1 = df_c1.iloc[idx_train].reset_index(drop=True)
df_val_c1   = df_c1.iloc[idx_val].reset_index(drop=True)
df_test_c1  = df_c1.iloc[idx_test].reset_index(drop=True)
Y_train = Y_c1[idx_train]
Y_val   = Y_c1[idx_val]
Y_test  = Y_c1[idx_test]

print(f"Split: train={len(df_train_c1)}, val={len(df_val_c1)}, test={len(df_test_c1)}")

active_overrides   = {k: v for k, v in COLUMN_TYPE_OVERRIDES.items() if k in feature_columns}
# IterativeImputer silently drops fully-empty columns and breaks the writeback
# in MICEImputer.transform_df, and indicator columns that are fully empty in
# train collapse to a constant. Filter both lists by training-split availability
# so the fit matches what's actually observable.
mice_cols      = [c for c in MICE_COLUMNS
                  if c in feature_columns and df_train_c1[c].notna().any()]
indicator_cols = [c for c in INDICATOR_COLUMNS
                  if c in feature_columns and df_train_c1[c].notna().any()]
_dropped_mice = [c for c in MICE_COLUMNS      if c in feature_columns and c not in mice_cols]
_dropped_ind  = [c for c in INDICATOR_COLUMNS if c in feature_columns and c not in indicator_cols]
if _dropped_mice or _dropped_ind:
    print(f"  MICE columns skipped (no train values): {_dropped_mice}")
    print(f"  Indicator columns skipped (no train values): {_dropped_ind}")

preprocessor = FeaturePreprocessor(
    preproc_config,
    column_types=active_overrides,
    mice_columns=mice_cols,
    indicator_columns=indicator_cols,
)

print("Fitting preprocessor on training split...")
print("=" * 60)
X_tab_train = preprocessor.fit_transform(df_train_c1[feature_columns].copy())
X_tab_val   = preprocessor.transform(df_val_c1[feature_columns].copy())
X_tab_test  = preprocessor.transform(df_test_c1[feature_columns].copy())

print()
print(f"Tabular feature shape (train): {X_tab_train.shape}")
print(f"Feature names ({len(preprocessor.get_feature_names())}):")
for name in preprocessor.get_feature_names():
    print(f"  - {name}")
if preprocessor.get_dropped_columns():
    print()
    print(f"Dropped columns: {preprocessor.get_dropped_columns()}")


## 3b. Morphological Feature Extraction

Before CNN feature extraction the AI-cleaned images are analysed with classical
image processing to produce **33 physically interpretable features** per image:

| Group | Features |
|-------|----------|
| Phase fractions | martensite_fraction, ferrite_fraction, phase_entropy |
| Ferrite geometry | grain_count, area mean/std/cv/skew/kurt, aspect_ratio, solidity, equiv_diam |
| Ferrite spatial | nearest-neighbour distance mean + std |
| Martensite topology | island_count, area_mean, aspect_ratio, connectivity, spacing |
| Boundary network | density, mean_width, banding_index |
| GLCM texture | contrast, energy, homogeneity, correlation, dissimilarity |
| LBP texture | entropy, uniformity |
| Intensity | local_contrast mean/std, intensity mean/std |

Images are loaded from the  column (AI-cleaned, annotation-free).
Failed images return NaN, which the preprocessor imputes with column medians.

See  for full design rationale and
literature references.


In [ ]:
# ── Initialise FeaturePipeline ───────────────────────────────────────────────
# All image download and CNN/morph extraction is handled by
# notebooks/prepare_features.ipynb (run once before this notebook).
# This cell only sets up the pipeline object used for loading.

DATA_DIR     = os.path.abspath('../data')
TEMP_DIR     = os.path.abspath('../data/temp_images')
FEATURES_DIR = os.path.abspath('../features')

fp = FeaturePipeline(
    data_dir=DATA_DIR,
    temp_dir=TEMP_DIR,
    features_dir=FEATURES_DIR,
)
fp.print_status()

# ── Force-rebuild dinov2_vitb14 + morph caches ───────────────────────────────
# This notebook is pinned to dinov2_vitb14 (see cell 24). We rebuild from raw
# images on every run so feature changes upstream propagate without manually
# clearing data/image_cache_dinov2_vitb14.npz or features/morph_features_c1.npz.
# Set FORCE_REBUILD = False to reuse the existing caches.
FORCE_REBUILD   = True
TARGET_BACKBONE = "dinov2_vitb14"

if FORCE_REBUILD:
    print(f"Rebuilding {TARGET_BACKBONE} cache (force=True)...")
    fp.extract_cnn(backbones=[TARGET_BACKBONE], force=True)
    print("Rebuilding morphological feature cache (force=True)...")
    fp.extract_morph(force=True)
    print("Rebuild complete.\n")
    fp.print_status()


In [ ]:
# ── Load morphological features ──────────────────────────────────────────────
# Cache is built by prepare_features.ipynb (§4). Loads and aligns to df_c1.

X_morph_train = X_morph_val = X_morph_test = None

_X_morph_all = fp.load_morph_features(df_c1["id"])
if _X_morph_all is not None:
    X_morph_train = _X_morph_all[idx_train]
    X_morph_val   = _X_morph_all[idx_val]
    X_morph_test  = _X_morph_all[idx_test]
    print(f'Morph features: {_X_morph_all.shape}')
    print(f'Splits: train={X_morph_train.shape}, val={X_morph_val.shape}, test={X_morph_test.shape}')
else:
    print('Morph cache not found -- run prepare_features.ipynb first.')
    print('Continuing without morphological features.')


## 4. Image Feature Extraction

The extraction module supports multiple CNN backbones:
- **Classic CNNs**: VGG16, VGG19, ResNet18/50/101
- **Modern Architectures**: DenseNet121, EfficientNet-B0/B4, ConvNeXt-Tiny
- **Self-supervised**: DINOv2 (ViT-S/B/L)

Image features are extracted from SEM images downloaded from Google Drive (see section 4b).

In [ ]:
# List available backbones
print("Available CNN Backbones:")
print("=" * 40)

available = BackboneRegistry.list_available()
for name in sorted(available):
    print(f"  - {name}")

print(f"\nTotal: {len(available)} backbones available")

In [ ]:
# Demonstrate extraction configuration (without actual images)
extraction_config = ExtractionConfig(
    backbones=['resnet50', 'vgg16'],  # Use multiple backbones
    img_size=224,
    batch_size=16,
    num_workers=2,
    pooling='avg'
)

print("Feature Extraction Configuration:")
print(f"  Backbones: {extraction_config.backbones}")
print(f"  Image size: {extraction_config.img_size}x{extraction_config.img_size}")
print(f"  Batch size: {extraction_config.batch_size}")
print(f"  Pooling: {extraction_config.pooling}")

# Show expected feature dimensions
backbone_dims = {
    'vgg16': 512, 'vgg19': 512,
    'resnet18': 512, 'resnet50': 2048, 'resnet101': 2048,
    'densenet121': 1024,
    'efficientnet_b0': 1280, 'efficientnet_b4': 1792,
    'convnext_tiny': 768,
    'mobilenet_v3': 960,
    'dinov2_vits14': 384, 'dinov2_vitb14': 768, 'dinov2_vitl14': 1024
}

total_dim = sum(backbone_dims.get(b, 0) for b in extraction_config.backbones)
print(f"\nExpected feature dimension: {total_dim}")

### 4b. Load Per-Backbone Feature Caches

> **Note:** CNN feature extraction is handled by `prepare_features.ipynb`.
> Run that notebook once to build all backbone caches. This cell only loads
> and aligns the selected backbone cache — it is fast and has no GPU requirement.


In [ ]:
# ── CNN cache status ─────────────────────────────────────────────────────────
# Caches are built by prepare_features.ipynb (§3).
# This cell just confirms what is available before cell 24 loads one.

from src.extraction.backbones import BackboneRegistry

ALL_BACKBONES = BackboneRegistry.list_available()
print(f'Registered backbones ({len(ALL_BACKBONES)}): {sorted(ALL_BACKBONES)}')

status = fp.verify()
print('\nCache status:')
for name, info in status["cnn"].items():
    mark = "ok" if info["exists"] else "MISSING"
    shape = str(info['shape']) if info['shape'] else ''
    print(f'  {name:<22} [{mark}]  {shape}')

print(f"\nDefault BACKBONE: set BACKBONE in cell 24 to one of the above.")


In [ ]:
# ── Load image features for selected backbone ────────────────────────────────
# Change BACKBONE to switch which CNN embeddings are used.
# Run prepare_features.ipynb §3 if a backbone cache is missing.

BACKBONE = 'dinov2_vitb14'  # pinned — this notebook only uses dinov2_vitb14

X_img_train = X_img_val = X_img_test = None

_X_img_all = fp.load_image_features(BACKBONE, df_c1["id"])
if _X_img_all is not None:
    X_img_train = _X_img_all[idx_train]
    X_img_val   = _X_img_all[idx_val]
    X_img_test  = _X_img_all[idx_test]
    print(f'Backbone: {BACKBONE}  |  feature_dim={_X_img_all.shape[1]}')
    print(f'Splits: train={X_img_train.shape}, val={X_img_val.shape}, test={X_img_test.shape}')
else:
    print(f'No cache for backbone={BACKBONE!r} -- using tabular features only.')
    print('Run prepare_features.ipynb to build CNN caches.')

print(f'Tabular: train={X_tab_train.shape}, val={X_tab_val.shape}, test={X_tab_test.shape}')


### 4c. Backbone Comparison

Trains ABR on `tabular + image` features for each available backbone cache and reports test R².  
Use results to choose `BACKBONE` in cell 21.


In [ ]:
# Backbone comparison — trains a fixed proxy regressor (from hyperparams.json,
# preference GBR > KNN > RF > ExtraTrees > XGB > ABR) once per cached backbone.
# Only runs backbones whose cache file exists (skips missing ones silently).
# Uses the same train/val/test split as the main pipeline for a fair comparison.

import os, gc
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score as _r2
import re as _re3
from collections import defaultdict

CACHE_DIR = os.path.abspath("../data")
_F_RE2    = _re3.compile(r'_F_\d+\.[a-z]+$', _re3.IGNORECASE)

def _align_cache(cache_path, df_c1, idx_train, idx_val, idx_test):
    """Load cache and return (X_train, X_val, X_test) aligned to df_c1."""
    data      = np.load(cache_path, allow_pickle=True)
    X_cache   = data["X"].astype(np.float32)
    filenames = list(data["filenames"])

    id_to_idx = defaultdict(list)
    for i, fname in enumerate(filenames):
        rid = _re3.sub(r'[-\s]+', '_', _F_RE2.sub('', os.path.basename(fname)).strip().lower())
        id_to_idx[rid].append(i)

    feat_dim = X_cache.shape[1]
    X_aligned = np.full((len(df_c1), feat_dim), np.nan, dtype=np.float32)
    for row_idx, row_id in enumerate(
            df_c1['id'].astype(str).str.strip().str.lower().str.replace(r'[-\s]+', '_', regex=True)):
        idxs = id_to_idx.get(row_id, [])
        if idxs:
            X_aligned[row_idx] = X_cache[idxs].mean(axis=0)

    col_means = np.nanmean(X_aligned, axis=0)
    nan_mask  = np.isnan(X_aligned).any(axis=1)
    X_aligned[nan_mask] = col_means
    return X_aligned[idx_train], X_aligned[idx_val], X_aligned[idx_test]

# ── Pick a stable proxy regressor from the hyperparam store ──────────────
# This cell runs in §4c (before the larger CV sweep in §6b), so it loads
# the manifest itself and defines a minimal local factory rather than
# depending on cell 37. Preference order favours fast, low-variance models.
from src import hyperparams as _hp_4c
from sklearn.ensemble import (RandomForestRegressor as _RF4, ExtraTreesRegressor as _ET4,
                              GradientBoostingRegressor as _GBR4)
from sklearn.neighbors import KNeighborsRegressor as _KNN4
from sklearn.svm import SVR as _SVR4
try:
    from xgboost import XGBRegressor as _XGB4
    _HAS_XGB4 = True
except ImportError:
    _HAS_XGB4 = False

_PROXY_PREFERENCE = ['GBR', 'XGB']  # debug notebook compares only these two
_HP_SCOPE_4C      = 'dp_steel'
_saved_params_4c  = _hp_4c.load(_HP_SCOPE_4C)

def _build_proxy_4c(name, params):
    p = dict(params)
    if name in ('RF', 'RandomForest'):
        return _RF4(**p, random_state=42, n_jobs=-1)
    if name == 'ExtraTrees':
        return _ET4(**p, random_state=42, n_jobs=-1)
    if name in ('GBR', 'GradientBoosting'):
        return MultiOutputRegressor(_GBR4(**p, random_state=42))
    if name in ('ABR', 'AdaBoost'):
        md = p.pop('max_depth', 3)
        return MultiOutputRegressor(AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=md), **p, random_state=42))
    if name == 'XGB' and _HAS_XGB4:
        return MultiOutputRegressor(_XGB4(**p, random_state=42, n_jobs=-1,
                                          tree_method='hist', verbosity=0))
    if name in ('KNN', 'KNeighbors'):
        return MultiOutputRegressor(_KNN4(**p))
    if 'SVR' in name:
        kernel = 'linear' if 'linear' in name.lower() else 'rbf'
        return MultiOutputRegressor(_SVR4(kernel=kernel, **p))
    raise ValueError(f'Unknown model in store: {name!r}')

def _make_proxy():
    for _n in _PROXY_PREFERENCE:
        if _saved_params_4c and _n in _saved_params_4c:
            _p = dict(_saved_params_4c[_n].get('params', {}))
            print(f"  proxy regressor: {_n} (from store, scope={_HP_SCOPE_4C})")
            return _build_proxy_4c(_n, _p), _n
    print("  proxy regressor: GBR (fallback default — store empty)")
    return MultiOutputRegressor(_GBR4(n_estimators=100, max_depth=3,
                                      random_state=42)), 'GBR'

_PROXY_TEMPLATE, _PROXY_NAME = _make_proxy()
from sklearn.base import clone as _clone_proxy

def _make_abr():
    # Name kept for back-compat with code below; returns a fresh proxy clone.
    return _clone_proxy(_PROXY_TEMPLATE)

results = {}  # backbone -> {val_r2, test_r2, feat_dim}

# Pinned to dinov2_vitb14 — see notebook header. Other backbones are skipped
# even if their caches exist on disk.
_BACKBONES_TO_COMPARE = ['dinov2_vitb14']
for backbone_name in _BACKBONES_TO_COMPARE:
    cache_path = os.path.join(CACHE_DIR, f'image_cache_{backbone_name}.npz')
    if not os.path.exists(cache_path):
        print(f"  skip {backbone_name}: cache missing at {cache_path}")
        continue

    Xi_tr, Xi_vl, Xi_te = _align_cache(cache_path, df_c1, idx_train, idx_val, idx_test)
    feat_dim = Xi_tr.shape[1]

    X_tr = np.concatenate([Xi_tr, X_tab_train], axis=1)
    X_vl = np.concatenate([Xi_vl, X_tab_val],   axis=1)
    X_te = np.concatenate([Xi_te, X_tab_test],   axis=1)

    model = _make_abr()
    model.fit(X_tr, Y_train)

    val_r2  = _r2(Y_val,  model.predict(X_vl), multioutput='uniform_average')
    test_r2 = _r2(Y_test, model.predict(X_te), multioutput='uniform_average')
    results[backbone_name] = {'val_r2': val_r2, 'test_r2': test_r2, 'feat_dim': feat_dim}
    del model; gc.collect()

# Tabular-only baseline
model_tab = _make_abr()
model_tab.fit(X_tab_train, Y_train)
tab_val  = _r2(Y_val,  model_tab.predict(X_tab_val),  multioutput='uniform_average')
tab_test = _r2(Y_test, model_tab.predict(X_tab_test), multioutput='uniform_average')
results['tabular_only'] = {'val_r2': tab_val, 'test_r2': tab_test, 'feat_dim': 0}

# ── Print table ───────────────────────────────────────────────────────────────
print(f"{'Backbone':<22} {'feat_dim':>9} {'val R²':>8} {'test R²':>9}")
print("-" * 52)
for name, m in sorted(results.items(), key=lambda x: -x[1]['test_r2']):
    marker = " ◀ best" if m['test_r2'] == max(v['test_r2'] for v in results.values()) else ""
    print(f"{name:<22} {m['feat_dim']:>9} {m['val_r2']:>8.4f} {m['test_r2']:>9.4f}{marker}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
names    = sorted(results, key=lambda x: -results[x]['test_r2'])
val_r2s  = [results[n]['val_r2']  for n in names]
test_r2s = [results[n]['test_r2'] for n in names]

x = np.arange(len(names))
w = 0.35
fig, ax = plt.subplots(figsize=(max(10, len(names) * 0.9), 5))
ax.bar(x - w/2, val_r2s,  w, label='Val R²',  alpha=0.8, color='steelblue')
ax.bar(x + w/2, test_r2s, w, label='Test R²', alpha=0.8, color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('R²')
ax.set_title(f'Backbone Comparison — proxy={_PROXY_NAME} (tabular + image features)')
ax.axhline(tab_test, color='red', linestyle='--', linewidth=1, label='Tabular-only baseline')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

best = max(results, key=lambda x: results[x]['test_r2'])
print(f"\nRecommended backbone: {best}  (test R²={results[best]['test_r2']:.4f})")
print(f"Set BACKBONE='{best}' in cell 21 to use it for training.")

## 5. Model Training & Evaluation

Multi-output regression with 2 targets (Cycle 1 HoldingTemp + HoldingTime):
- **Random Forest**: Native multi-output support
- **Gradient Boosting**: Wrapped in MultiOutputRegressor
- **AdaBoost**: Wrapped in MultiOutputRegressor

**Note**: Learning curves (`staged_predict`) are only available for single-output boosting models, so they are skipped for this multi-output task.

In [ ]:
# Targets are already split (Y_train/Y_val/Y_test set in cell 13)
Y = Y_c1  # full array kept for reference

print(f"Target columns: {target_columns}")
for i, name in enumerate(target_columns):
    print(f"  {name}: range=[{Y[:, i].min():.1f}, {Y[:, i].max():.1f}], "
          f"mean={Y[:, i].mean():.1f}, std={Y[:, i].std():.1f}")
print()
print(f"Split sizes: train={len(Y_train)}, val={len(Y_val)}, test={len(Y_test)}")


In [ ]:
# Combine CNN image, morphological, and tabular features per split.
# Each stream is optional — the cell degrades gracefully if any is absent.

parts_train, parts_val, parts_test = [], [], []
dim_log = []

if X_img_train is not None:
    parts_train.append(X_img_train)
    parts_val.append(X_img_val)
    parts_test.append(X_img_test)
    dim_log.append(f"{X_img_train.shape[1]} (CNN image)")

if X_morph_train is not None:
    parts_train.append(X_morph_train)
    parts_val.append(X_morph_val)
    parts_test.append(X_morph_test)
    dim_log.append(f"{X_morph_train.shape[1]} (morphological)")

parts_train.append(X_tab_train)
parts_val.append(X_tab_val)
parts_test.append(X_tab_test)
dim_log.append(f"{X_tab_train.shape[1]} (tabular)")

X_train = np.concatenate(parts_train, axis=1)
X_val   = np.concatenate(parts_val,   axis=1)
X_test  = np.concatenate(parts_test,  axis=1)

print("Feature streams combined:")
print("  " + " + ".join(dim_log) + f" = {X_train.shape[1]} total")
print(f"Shapes — train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")

# ── Impute NaN rows from missing image/morph caches ────────────────────
# X_img and X_morph return NaN for IDs without cached features; GBR/ABR/
# XGB don't accept NaN. Fit median imputer on train only (no leakage),
# apply to all three splits. Skip silently if there's nothing to impute.
if np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any():
    from sklearn.impute import SimpleImputer
    _imp = SimpleImputer(strategy="median")
    X_train = _imp.fit_transform(X_train)
    X_val   = _imp.transform(X_val)
    X_test  = _imp.transform(X_test)
    print(f"Imputed NaN rows from missing image/morph caches "
          f"(median per feature, fit on train only).")


In [ ]:
# Create a minimal config for the trainer
config = Config(
    random_seed=42,
    model_dir='../models'
)

# Initialize trainer with configurable n_estimators
trainer = ModelTrainer(config, n_estimators=100)  # Reduced for faster demo

print(f"Model Trainer initialized:")
print(f"  Random seed: {config.random_seed}")
print(f"  N estimators: {trainer.n_estimators}")
print(f"  Model directory: {config.model_dir}")

In [ ]:
# Data is already split in cell 13 — no need to re-split here.
print(f"Data splits:")
print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")


In [ ]:
# Train all models and track learning curves
print("\nTraining ensemble models...")
print("=" * 60)

best_model_name = trainer.train_and_evaluate(
    X_train, Y_train,
    X_val, Y_val,
    target_columns,
    track_learning_curves=True  # Enable learning curve tracking
)

In [ ]:
# Learning curves are not available for multi-output models (staged_predict requires single output)
# This cell will report "No learning histories" which is expected behavior
print("Learning Curves:")
print("=" * 60)
print("Skipped: staged_predict is not available for multi-output boosting models.")
print("Learning curves are only supported when n_targets == 1.")

In [ ]:
# Evaluate all models on test set
test_results = trainer.evaluate_all_on_test(X_test, Y_test, target_columns)

In [ ]:
# Plot model comparison
trainer.plot_test_comparison(show=True)
# Save model comparison figure to run directory
try:
    import matplotlib.pyplot as _plt_rc
    _plt_rc.savefig(str(_run_dir / "model_comparison.png"), dpi=150, bbox_inches="tight")
except Exception:
    pass


## 6b. Model Improvement Techniques

Three techniques applied to improve reliability on this small dataset:
- **Cross-validation** — more stable R² estimate than a single val split; model params loaded from store when available
- **Load from store** — best model hyperparams and preprocessing config loaded from  and  (written by  / )
- **Feature importance pruning** — remove low-signal features that add noise
- **SMOGN tabular augmentation** — synthesise new samples for underrepresented target ranges


In [ ]:
# ── Load saved hyperparams (written by bayes_tuning.ipynb) ───────────────────
# Build the CV sweep dynamically from whatever models the latest tuning run
# produced — no hardcoded {RF, GBR, ABR, ExtraTrees}. If a tuning run added
# (or dropped) a model, this cell tracks it.
_HP_SCOPE     = "dp_steel"
_saved_params = hp.load(_HP_SCOPE)
if _saved_params:
    print(f"Loaded saved hyperparams for {len(_saved_params)} model(s) "
          f"(scope={_HP_SCOPE}): {list(_saved_params.keys())}")
else:
    print(f"No saved hyperparams found for scope={_HP_SCOPE} — "
          f"run bayes_tuning.ipynb first or accept the empty CV sweep.")

# ── CV protocol ───────────────────────────────────────────────────────────────
# With ~100 samples a single val split can swing ±0.1 R² depending on which
# 13 samples land in it. RepeatedKFold gives a much more reliable estimate.
from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (AdaBoostRegressor, RandomForestRegressor,
                               GradientBoostingRegressor, ExtraTreesRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, r2_score as _r2_score
try:
    from xgboost import XGBRegressor
    _XGB_OK = True
except ImportError:
    _XGB_OK = False

_multi_r2 = make_scorer(_r2_score, multioutput="uniform_average")

# Use the full Cycle 1 feature matrix (no test hold-out — CV handles it).
X_full = np.concatenate([X_img_train, X_tab_train], axis=1) if X_img_train is not None else X_tab_train
X_full_all = np.vstack([X_full,
                        (np.concatenate([X_img_val, X_tab_val], axis=1) if X_img_val is not None else X_tab_val),
                        (np.concatenate([X_img_test, X_tab_test], axis=1) if X_img_test is not None else X_tab_test)])
Y_full_all = np.vstack([Y_train, Y_val, Y_test])

cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)


# ─────────────────────────────────────────────────────────────────────────────
# Shared factory: build any supported regressor from a stored params dict.
# Reused by cell 38. Keep this in cell 37 so downstream cells inherit it.
# ─────────────────────────────────────────────────────────────────────────────
def _build_from_params(name: str, params: dict):
    """Construct an sklearn-compatible estimator for *name* using *params*.

    Mirrors the search space defined in bayes_tuning.ipynb's build_model().
    Wraps single-output learners in MultiOutputRegressor so they can predict
    both HoldingTemp and HoldingTime in one shot.
    """
    p = dict(params)  # don't mutate caller's dict
    base = name.replace("_pruned", "")

    if base in ("RF", "RandomForest"):
        return RandomForestRegressor(**p, random_state=42, n_jobs=-1)

    if base == "ExtraTrees":
        return ExtraTreesRegressor(**p, random_state=42, n_jobs=-1)

    if base in ("GBR", "GradientBoosting"):
        return MultiOutputRegressor(GradientBoostingRegressor(**p, random_state=42))

    if base in ("ABR", "AdaBoost"):
        md = p.pop("max_depth", 3)
        return MultiOutputRegressor(AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=md),
            **p, random_state=42))

    if base == "XGB":
        if not _XGB_OK:
            raise RuntimeError("XGBoost not installed but XGB params loaded from store.")
        return MultiOutputRegressor(XGBRegressor(
            **p, random_state=42, n_jobs=-1, tree_method='hist',
            verbosity=0))

    if base in ("KNN", "KNeighbors"):
        return MultiOutputRegressor(KNeighborsRegressor(**p))

    if "SVR" in base or base == "SVM":
        kernel = "linear" if "linear" in base.lower() else "rbf"
        return MultiOutputRegressor(SVR(kernel=kernel, **p))

    raise ValueError(f"Unknown model name in store: {name!r}")


# ── Build cv_models — debug notebook restricts to {GBR, XGB} only ────────────
# This notebook intentionally compares two regressors. Use store params when
# the manifest has them; otherwise fall back to sensible defaults so the sweep
# still runs without bayes_tuning being executed first.
_DEBUG_REGRESSORS = ['GBR', 'XGB']
cv_models = {}
for _name in _DEBUG_REGRESSORS:
    if _saved_params and _name in _saved_params:
        _params = _saved_params[_name].get('params', {})
        try:
            cv_models[_name] = _build_from_params(_name, _params)
            print(f"  {_name:<12} [store]    params={_params}")
        except Exception as e:
            print(f"  {_name:<12} [skipped]  {e}")
    else:
        # Defaults match bayes_tuning.ipynb's search-space midpoints.
        if _name == 'GBR':
            cv_models[_name] = MultiOutputRegressor(GradientBoostingRegressor(
                n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42))
            print(f"  {_name:<12} [default]  n_estimators=200, max_depth=3, lr=0.05")
        elif _name == 'XGB':
            if not _XGB_OK:
                print(f"  {_name:<12} [skipped]  xgboost not installed")
                continue
            cv_models[_name] = MultiOutputRegressor(XGBRegressor(
                n_estimators=200, max_depth=4, learning_rate=0.05,
                random_state=42, n_jobs=-1, tree_method='hist', verbosity=0))
            print(f"  {_name:<12} [default]  n_estimators=200, max_depth=4, lr=0.05")

cv_scores = {}  # name -> score array (consumed by metrics logger downstream)
print(f"\nCross-validation results (RepeatedKFold 5x10) on {X_full_all.shape}:")
print("=" * 60)
for _name, _model in cv_models.items():
    cv_scores[_name] = cross_val_score(_model, X_full_all, Y_full_all, cv=cv,
                                        scoring=_multi_r2, n_jobs=-1)
    print(f"  {_name:<12} R² = {cv_scores[_name].mean():.4f} ± {cv_scores[_name].std():.4f}  "
          f"(min={cv_scores[_name].min():.3f}, max={cv_scores[_name].max():.3f})")


In [ ]:
# ── Pick the best model from the hyperparam store + benchmark results ───────
# Hyperparams come from bayes_tuning.ipynb → runs/hyperparams.json (scope=dp_steel).
# Best preprocessing / regressor come from pipeline_benchmark.ipynb →
# runs/benchmark_results.csv. _build_from_params() is defined in cell 37.

import os as _os
import pandas as _pd
from sklearn.base import clone as _clone

# ── 1. Hyperparam store (already loaded in cell 37 as _saved_params) ──────────
if _saved_params:
    print(f"Hyperparam store loaded for scope=\"{_HP_SCOPE}\":")
    for _n, _v in _saved_params.items():
        _r2 = _v.get("tuned_cv_r2", _v.get("best_cv_r2", "?"))
        print(f"  {_n:<14}  tuned_cv_r2={_r2}")
else:
    print(f"No saved hyperparams for scope=\"{_HP_SCOPE}\" — falling back to CV winner.")

# ── 2. Optional benchmark results (preprocessing/regressor winners) ──────────
_bench_path = _os.path.join(_os.path.dirname(_os.path.abspath("runs")), "runs", "benchmark_results.csv")
_bench_model   = None
_bench_preproc = None
if _os.path.exists(_bench_path):
    _df_bench = _pd.read_csv(_bench_path)
    _reg_rows = _df_bench[_df_bench["stage"] == "regressor"]
    _pre_rows = _df_bench[_df_bench["stage"] == "preprocessing"]
    if not _reg_rows.empty:
        _bench_model = _reg_rows.sort_values("mean_r2", ascending=False).iloc[0]["config"]
    if not _pre_rows.empty:
        _bench_preproc = _pre_rows.sort_values("mean_r2", ascending=False).iloc[0]["config"]
    print(f"Benchmark results: best regressor={_bench_model}  best preprocessing={_bench_preproc}")

# ── 3. Resolve which model to use ─────────────────────────────────────────────
# Priority:
#   (a) benchmark winner that's also in the store
#   (b) store winner (hp.best_model)
#   (c) CV winner from cell 37 (cv_scores)
#   (d) hardcoded RF fallback
_use_name = None
if _bench_model and _bench_model in _saved_params:
    _use_name = _bench_model
elif _saved_params:
    _use_name, _ = hp.best_model(_HP_SCOPE)
elif cv_scores:
    _use_name = max(cv_scores, key=lambda k: cv_scores[k].mean())
else:
    _use_name = "RF"

_use_params = dict(_saved_params.get(_use_name, {}).get("params", {})) if _saved_params else {}
print(f"Selected model    : {_use_name}")
print(f"Source of params  : {'store' if _use_params else 'defaults'}")

# ── 4. Build the chosen estimator (factory comes from cell 37) ────────────────
if _use_params:
    _best_estimator = _build_from_params(_use_name, _use_params)
else:
    _best_estimator = RandomForestRegressor(n_estimators=200, max_features="sqrt",
                                             min_samples_leaf=3, random_state=42, n_jobs=-1)

# ── 5. Cross-validate the chosen model ────────────────────────────────────────
CV_FULL = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
print(f"\nCross-validating {_use_name} (RepeatedKFold 5×10)...")
_full_sc    = cross_val_score(_best_estimator, X_full_all, Y_full_all,
                               cv=CV_FULL, scoring=_multi_r2, n_jobs=-1)
_untuned_r2 = cv_scores.get(_use_name, _full_sc).mean()
_delta      = _full_sc.mean() - _untuned_r2 if _use_name in cv_scores else 0.0
print(f"  CV R² = {_full_sc.mean():.4f} ± {_full_sc.std():.4f}  "
      f"(delta vs cell-37 sweep: {_delta:+.4f})")

# ── 6. Set variables consumed downstream (predictions, metrics logger) ───────
best_tuned_name = _use_name
best_tuned_r2   = float(_full_sc.mean())
bayes_eval = {
    _use_name: {
        "untuned_r2": float(_untuned_r2),
        "tuned_r2":   best_tuned_r2,
        "tuned_std":  float(_full_sc.std()),
        "delta":      float(_delta),
        "params":     _use_params,
    }
}

# Fit on train split so prediction cells can reuse it.
best_tuned_model = _clone(_best_estimator)
best_tuned_model.fit(X_train, Y_train)

print(f"\nbest_tuned_name = {best_tuned_name}")
print(f"best_tuned_r2   = {best_tuned_r2:.4f}")
if _bench_preproc:
    print(f"Benchmark preprocessing: {_bench_preproc}")


In [ ]:
# ── Feature importance pruning ─────────────────────────────────────────────
# Use feature importances from a manifest-driven tree model to derive a
# keep_mask (bottom 20% dropped), then compare full vs pruned across cv_models.
#
# Importance source picked from _saved_params (cell 37) using preference order
# below. KNN and SVR don't expose feature_importances_, so we walk past them.
# If nothing usable is in the store, the whole cell is skipped — we don't fall
# back to a hardcoded RF since that would report importances from an untuned
# model and silently mislead anyone reading the resulting bar chart.
import copy
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score as _r2

X_tr = np.concatenate([X_img_train, X_tab_train], axis=1) if X_img_train is not None else X_tab_train
X_vl = np.concatenate([X_img_val,   X_tab_val],   axis=1) if X_img_val  is not None else X_tab_val
X_te = np.concatenate([X_img_test,  X_tab_test],  axis=1) if X_img_test is not None else X_tab_test

# Tree-ish models that expose feature_importances_, ranked by usefulness here.
_IMP_PREFERENCE = ['GBR', 'XGB', 'ExtraTrees', 'RF', 'ABR']
_imp_name       = next((n for n in _IMP_PREFERENCE if n in (_saved_params or {})), None)

if _imp_name is None:
    print(f"No tree-based model in _saved_params (have: "
          f"{list((_saved_params or {}).keys())}); skipping feature-importance cell. "
          f"Re-run bayes_tuning with one of {_IMP_PREFERENCE} included to enable.")
else:
    # ── Step 1: derive importance mask from the chosen model ────────────────
    _imp_params = dict(_saved_params[_imp_name].get("params", {}))
    _imp_model  = _build_from_params(_imp_name, _imp_params)
    print(f"Importance source: {_imp_name} (params from store)")
    _imp_model.fit(X_tr, Y_train)

    # MultiOutputRegressor wraps GBR/XGB/ABR for the dual-target Y; pull the
    # per-target importances and average them so the mask is target-agnostic.
    if hasattr(_imp_model, "estimators_"):
        per_target_imp = np.vstack([est.feature_importances_
                                    for est in _imp_model.estimators_])
        importances = per_target_imp.mean(axis=0)
    else:
        importances = _imp_model.feature_importances_

    threshold = np.percentile(importances, 20)
    keep_mask = importances >= threshold
    print(f"Keeping {keep_mask.sum()}/{len(importances)} features (dropped bottom 20%)")

    # ── Step 2: full vs pruned test R² for all cv_models ────────────────────
    print("\nPruning comparison — test R² (uniform average)")
    print(f"{'Model':<14} {'Full':>8} {'Pruned':>8} {'Delta':>8}")
    print("-" * 44)
    for _name, _model in cv_models.items():
        _m_full   = copy.deepcopy(_model)
        _m_pruned = copy.deepcopy(_model)
        _m_full.fit(X_tr, Y_train)
        _m_pruned.fit(X_tr[:, keep_mask], Y_train)
        r2_full   = _r2(Y_test, _m_full.predict(X_te),                 multioutput="uniform_average")
        r2_pruned = _r2(Y_test, _m_pruned.predict(X_te[:, keep_mask]), multioutput="uniform_average")
        print(f"  {_name:<12} {r2_full:8.4f} {r2_pruned:8.4f} {r2_pruned - r2_full:+8.4f}")

    # ── Step 3: top-20 feature importance bar chart ─────────────────────────
    feat_names = (["img_" + str(i) for i in range(X_tr.shape[1] - X_tab_train.shape[1])]
                  if X_img_train is not None else []) + list(preprocessor.get_feature_names())
    top20_idx = np.argsort(importances)[-20:][::-1]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(range(20), importances[top20_idx])
    ax.set_xticks(range(20))
    ax.set_xticklabels([feat_names[i] if i < len(feat_names) else f"f{i}"
                        for i in top20_idx], rotation=45, ha="right", fontsize=8)
    ax.set_title(f"Top 20 Feature Importances ({_imp_name} — used for pruning mask)")
    ax.set_ylabel("Importance")
    plt.tight_layout()
    plt.show()


In [ ]:
# ── SMOGN tabular augmentation ────────────────────────────────────────────────
# Synthesises new tabular samples for underrepresented target ranges, then
# checks whether the augmentation changes test R². The evaluator model is
# picked from _saved_params (cell 37) — same preference order as cell 39 —
# so the test reflects the model the rest of the notebook actually uses.
#
# SMOGN itself only operates on tabular features, so both baseline and
# augmented runs use X_tab_* (no image features). Skips cleanly when SMOGN
# isn't installed or when no tree-based model is in the store.
try:
    import smogn
    import pandas as pd

    # SMOGN operates on a single target — run per target, take the larger
    # augmented set (more conservative — bigger variance reduction).
    df_aug_input = pd.DataFrame(X_tab_train,
                                columns=preprocessor.get_feature_names())
    df_aug_input[c1_temp_col] = Y_train[:, 0]
    df_aug_input[c1_time_col] = Y_train[:, 1]

    df_aug_temp = smogn.smoter(data=df_aug_input, y=c1_temp_col, k=3,
                                samp_method='balance', random_state=42)
    df_aug_time = smogn.smoter(data=df_aug_input, y=c1_time_col, k=3,
                                samp_method='balance', random_state=42)

    df_aug = df_aug_temp if len(df_aug_temp) >= len(df_aug_time) else df_aug_time
    feat_cols = preprocessor.get_feature_names()
    X_tab_train_aug = df_aug[feat_cols].values.astype(np.float32)
    Y_train_aug     = df_aug[[c1_temp_col, c1_time_col]].values.astype(np.float64)

    print(f"SMOGN augmentation: {len(Y_train)} -> {len(Y_train_aug)} training samples")

    # ── Pick the evaluator from the hyperparam store ────────────────────────
    # Prefer tree-based models (richer fit on tabular). KNN/SVR are valid too
    # but less informative for testing whether augmentation helps a real model.
    _SMOGN_PREFERENCE = ['GBR', 'XGB', 'ExtraTrees', 'RF', 'ABR', 'KNN']
    _eval_name = next((n for n in _SMOGN_PREFERENCE if n in (_saved_params or {})), None)

    if _eval_name is None:
        print(f"No usable model in _saved_params (have: "
              f"{list((_saved_params or {}).keys())}); skipping SMOGN evaluation. "
              f"Re-run bayes_tuning with one of {_SMOGN_PREFERENCE} included.")
    else:
        from sklearn.base import clone as _clone_smogn
        _eval_params = dict(_saved_params[_eval_name].get("params", {}))
        _eval_template = _build_from_params(_eval_name, _eval_params)
        print(f"Evaluator: {_eval_name} (params from store)")

        # Apples-to-apples: same model, same tabular-only feature set, only
        # the training data differs (original vs SMOGN-augmented).
        _m_base = _clone_smogn(_eval_template)
        _m_aug  = _clone_smogn(_eval_template)
        _m_base.fit(X_tab_train,     Y_train)
        _m_aug.fit (X_tab_train_aug, Y_train_aug)
        r2_base = _r2(Y_test, _m_base.predict(X_tab_test), multioutput='uniform_average')
        r2_aug  = _r2(Y_test, _m_aug.predict (X_tab_test), multioutput='uniform_average')

        print(f"{_eval_name} baseline (no aug)  test R2: {r2_base:.4f}")
        print(f"{_eval_name} + SMOGN aug        test R2: {r2_aug:.4f}")
        print(f"Delta from augmentation:        {r2_aug - r2_base:+.4f}")

except ImportError:
    print("smogn not installed -- run: pip install smogn")
    print("Skipping augmentation cell.")


## 6. Predictions Visualization

Per-target predicted vs actual plots and residual analysis for the multi-output model.

In [ ]:
# ── Top-3 model predictions ───────────────────────────────────────────────────
# Rank cv_models by mean CV R², pick top 3, fit each on train, predict on test.
import copy
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

_ranked = sorted(cv_scores.items(), key=lambda kv: kv[1].mean(), reverse=True)
_top3_names = [k for k, _ in _ranked[:3]]
print("Top 3 models by CV R²:")
for _n, _sc in _ranked[:3]:
    print(f"  {_n:<14} CV R² = {_sc.mean():.4f} ± {_sc.std():.4f}")

# Fit each top-3 model on the train split, predict on test
top3_preds   = {}   # name -> Y_pred (n_test × n_targets)
top3_metrics = {}   # name -> {R2, MAE, RMSE}
for _n in _top3_names:
    _m = copy.deepcopy(cv_models[_n])
    _m.fit(X_tr, Y_train)
    _yp = _m.predict(X_te)
    if _yp.ndim == 1:
        _yp = _yp.reshape(-1, 1)
    top3_preds[_n] = _yp
    top3_metrics[_n] = {
        "R2":   r2_score(Y_test,  _yp, multioutput="uniform_average"),
        "MAE":  mean_absolute_error(Y_test, _yp, multioutput="uniform_average"),
        "RMSE": float(np.sqrt(mean_squared_error(Y_test, _yp, multioutput="uniform_average"))),
    }

print("\nTest set metrics (top 3):")
print(f"  {'Model':<14} {'R²':>8} {'MAE':>10} {'RMSE':>10}")
print("  " + "-" * 44)
for _n in _top3_names:
    m = top3_metrics[_n]
    print(f"  {_n:<14} {m['R2']:8.4f} {m['MAE']:10.4f} {m['RMSE']:10.4f}")

# Best of the three (for downstream residual / summary cells)
_best_name = max(top3_metrics, key=lambda k: top3_metrics[k]["R2"])
Y_pred      = top3_preds[_best_name]
print(f"\nBest of top 3: {_best_name}  (R² = {top3_metrics[_best_name]['R2']:.4f})")


In [ ]:
# ── Predicted vs actual — top 3 models, one row per model ────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

n_targets = len(target_columns)
fig, axes = plt.subplots(len(_top3_names), n_targets,
                         figsize=(6 * n_targets, 5 * len(_top3_names)),
                         squeeze=False)

for row, _n in enumerate(_top3_names):
    _yp = top3_preds[_n]
    for col_idx, col in enumerate(target_columns):
        ax = axes[row][col_idx]
        y_true = Y_test[:, col_idx]
        y_pred = _yp[:, col_idx]

        ax.scatter(y_true, y_pred, alpha=0.65, edgecolors="white", linewidth=0.5, s=60)
        mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
        ax.plot([mn, mx], [mn, mx], "r--", linewidth=2, label="Perfect")
        r2 = r2_score(y_true, y_pred)
        ax.annotate(f"R² = {r2:.4f}", xy=(0.05, 0.92), xycoords="axes fraction",
                    fontsize=11, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
        ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        ax.set_title(f"{col}  [{_n}]", fontsize=11)
        ax.legend(fontsize=9)

plt.suptitle("Predicted vs Actual — Top 3 CV models", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Residual analysis — best of top 3 ────────────────────────────────────────
import matplotlib.pyplot as plt

n_targets = len(target_columns)
fig, axes = plt.subplots(n_targets, 2, figsize=(12, 4 * n_targets))
if n_targets == 1:
    axes = axes.reshape(1, -1)

for i, col in enumerate(target_columns):
    residuals = Y_test[:, i] - Y_pred[:, i]

    axes[i, 0].hist(residuals, bins=15, color="steelblue", alpha=0.7, edgecolor="white")
    axes[i, 0].axvline(x=0, color="red", linestyle="--", linewidth=2)
    axes[i, 0].set_xlabel("Residual"); axes[i, 0].set_ylabel("Frequency")
    axes[i, 0].set_title(f"{col}: Residual Distribution [{_best_name}]")

    axes[i, 1].scatter(Y_pred[:, i], residuals, alpha=0.6)
    axes[i, 1].axhline(y=0, color="red", linestyle="--", linewidth=2)
    axes[i, 1].set_xlabel("Predicted"); axes[i, 1].set_ylabel("Residuals")
    axes[i, 1].set_title(f"{col}: Residuals vs Predicted [{_best_name}]")

    print(f"{col} residuals ({_best_name}): mean={residuals.mean():.2f}, "
          f"std={residuals.std():.2f}, min={residuals.min():.2f}, max={residuals.max():.2f}")

plt.tight_layout()
plt.show()


In [ ]:
# ── Final summary — top 3 comparison ─────────────────────────────────────────
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

n_samples = len(df_c1)
print("=" * 60)
print("CYCLE 1 MODEL SUMMARY — TOP 3 REGRESSORS")
print("=" * 60)
print(f"Dataset : {n_samples} samples")
print(f"Targets : {target_columns} ({len(target_columns)} outputs)")
print(f"Tabular : {X_tab_train.shape[1]} features (chemical composition)")
image_dim = X_img_train.shape[1] if X_img_train is not None else 0
print(f"Image   : {image_dim} features ({'from cache' if image_dim > 0 else 'not available'})")
print(f"Total   : {X_te.shape[1]} features")
print()

print(f"{'Model':<14} {'R²':>8} {'MAE':>10} {'RMSE':>10}  per-target R²")
print("-" * 65)
for _n in _top3_names:
    _yp = top3_preds[_n]
    m   = top3_metrics[_n]
    _pt = "  ".join(f"{r2_score(Y_test[:, i], _yp[:, i]):.3f}" for i in range(len(target_columns)))
    _marker = "  ◀ best" if _n == _best_name else ""
    print(f"  {_n:<12} {m['R2']:8.4f} {m['MAE']:10.4f} {m['RMSE']:10.4f}  [{_pt}]{_marker}")

print()
print(f"Best model : {_best_name}")
print(f"Test R²    : {top3_metrics[_best_name]['R2']:.4f}")
print(f"Test MAE   : {top3_metrics[_best_name]['MAE']:.4f}")
print("=" * 60)


## 7. Extension: Cycle 1 + Cycle 2 Multi-Output Prediction

52 samples have both Cycle 1 and Cycle 2 targets available. This section trains a 4-output model:
- Cycle1_HoldingTemp, Cycle1_HoldingTime
- Cycle2_HoldingTemp, Cycle2_HoldingTime

**Tradeoff**: More targets but fewer samples (52 vs 82).

In [ ]:
# Filter to samples with all 4 targets (Cycle 1 + Cycle 2) available
c2_temp_col = target_col_map["Cycle2_HoldingTemp"]
c2_time_col = target_col_map["Cycle2_HoldingTime"]

c1c2_mask = (
    df[c1_temp_col].notna() & df[c1_time_col].notna() &
    df[c2_temp_col].notna() & df[c2_time_col].notna()
)
df_c1c2 = df[c1c2_mask].copy().reset_index(drop=True)

target_columns_ext = [c1_temp_col, c1_time_col, c2_temp_col, c2_time_col]
Y_c1c2 = df_c1c2[target_columns_ext].values.astype(np.float64)

print(f"Cycle 1+2 subset: {len(df_c1c2)} samples")
print(f"Target columns: {target_columns_ext}")
for i, col in enumerate(target_columns_ext):
    vals = Y_c1c2[:, i]
    print(f"  {col}: range=[{vals.min():.1f}, {vals.max():.1f}], mean={vals.mean():.1f}, std={vals.std():.1f}")

In [ ]:
# Preprocess features for the Cycle 1+2 subset
# Split FIRST, then fit preprocessor on train split only.
from sklearn.model_selection import train_test_split

idx_ext = np.arange(len(df_c1c2))
idx_ext_trainval, idx_ext_test = train_test_split(idx_ext, test_size=0.15, random_state=42)
idx_ext_train, idx_ext_val     = train_test_split(idx_ext_trainval, test_size=0.15, random_state=42)

df_ext_train = df_c1c2.iloc[idx_ext_train].reset_index(drop=True)
df_ext_val   = df_c1c2.iloc[idx_ext_val].reset_index(drop=True)
df_ext_test  = df_c1c2.iloc[idx_ext_test].reset_index(drop=True)
Y_train_ext  = Y_c1c2[idx_ext_train]
Y_val_ext    = Y_c1c2[idx_ext_val]
Y_test_ext   = Y_c1c2[idx_ext_test]

# Tabular features
active_overrides_ext = {k: v for k, v in COLUMN_TYPE_OVERRIDES.items() if k in feature_columns}
# Same filter as cell 15 — drop MICE/indicator cols that are fully empty in this
# subset's train split, otherwise IterativeImputer silently strips them and
# MICEImputer.transform_df can't write back the missing columns.
mice_cols_ext      = [c for c in MICE_COLUMNS
                      if c in feature_columns and df_ext_train[c].notna().any()]
indicator_cols_ext = [c for c in INDICATOR_COLUMNS
                      if c in feature_columns and df_ext_train[c].notna().any()]

preprocessor_ext = FeaturePreprocessor(
    preproc_config,
    column_types=active_overrides_ext,
    mice_columns=mice_cols_ext,
    indicator_columns=indicator_cols_ext,
)
X_tab_train_ext = preprocessor_ext.fit_transform(df_ext_train[feature_columns].copy())
X_tab_val_ext   = preprocessor_ext.transform(df_ext_val[feature_columns].copy())
X_tab_test_ext  = preprocessor_ext.transform(df_ext_test[feature_columns].copy())

# Morphological features for the Cycle 1+2 subset ──────────────────────────────
# Reuse the same key_to_paths index built in cell 17.
X_morph_train_ext = X_morph_val_ext = X_morph_test_ext = None

if all_imgs:  # all_imgs defined in cell 17
    MORPH_CACHE_EXT = os.path.abspath('../features/morph_features_c1c2.npz')
    morph_names_ext = morph_extractor.get_feature_names()
    n_feats_ext     = len(morph_names_ext)

    if os.path.exists(MORPH_CACHE_EXT):
        print(f'Loading Cycle 1+2 morphological cache: {MORPH_CACHE_EXT}')
        _cd2 = np.load(MORPH_CACHE_EXT, allow_pickle=True)
        X_morph_ext_all = _cd2['X'].astype(np.float64)
    else:
        X_morph_ext_all = np.full((len(df_c1c2), n_feats_ext), np.nan, dtype=np.float64)
        for row_idx, row_id in enumerate(df_c1c2['id']):
            paths = key_to_paths.get(_id_key(row_id), [])
            if not paths:
                continue
            row_feats = np.vstack([morph_extractor.extract_single(p) for p in paths])
            X_morph_ext_all[row_idx] = np.nanmean(row_feats, axis=0)
        os.makedirs(os.path.dirname(MORPH_CACHE_EXT), exist_ok=True)
        np.savez(MORPH_CACHE_EXT, X=X_morph_ext_all)
        print(f'Cached to {MORPH_CACHE_EXT}')

    df_morph_ext       = pd.DataFrame(X_morph_ext_all, columns=morph_names_ext)
    df_morph_ext_train = df_morph_ext.iloc[idx_ext_train].reset_index(drop=True)
    df_morph_ext_val   = df_morph_ext.iloc[idx_ext_val].reset_index(drop=True)
    df_morph_ext_test  = df_morph_ext.iloc[idx_ext_test].reset_index(drop=True)

    morph_preproc_ext = FeaturePreprocessor(
        PreprocessingConfig(
            missing_data=MissingDataConfig(
                column_drop_threshold=0.95,
                row_fill_threshold=1.0,
                numeric_fill_strategy='median',
                categorical_fill_strategy='mode',
            ),
            scaling=ScalingConfig(method='standard', enabled=True),
            encoding=EncodingConfig(categorical='onehot', max_categories=50),
        )
    )
    X_morph_train_ext = morph_preproc_ext.fit_transform(df_morph_ext_train)
    X_morph_val_ext   = morph_preproc_ext.transform(df_morph_ext_val)
    X_morph_test_ext  = morph_preproc_ext.transform(df_morph_ext_test)
    print(f'Morphological features (Cycle 1+2 subset): {X_morph_train_ext.shape[1]}')

# Combine
parts_ext      = [X_tab_train_ext]
parts_ext_val  = [X_tab_val_ext]
parts_ext_test = [X_tab_test_ext]
dim_ext_log    = [f'{X_tab_train_ext.shape[1]} (tabular)']

if X_morph_train_ext is not None:
    parts_ext.insert(0, X_morph_train_ext)
    parts_ext_val.insert(0, X_morph_val_ext)
    parts_ext_test.insert(0, X_morph_test_ext)
    dim_ext_log.insert(0, f'{X_morph_train_ext.shape[1]} (morphological)')

X_train_ext = np.concatenate(parts_ext,      axis=1)
X_val_ext   = np.concatenate(parts_ext_val,  axis=1)
X_test_ext  = np.concatenate(parts_ext_test, axis=1)

print(f'Cycle 1+2 splits: train={len(df_ext_train)}, val={len(df_ext_val)}, test={len(df_ext_test)}')
print(f'Features: {" + ".join(dim_ext_log)} = {X_train_ext.shape[1]} total')
print(f'Target matrix (train):  {Y_train_ext.shape}')

# ── Impute any residual NaNs (same pattern as cell 29) ─────────────────
# Tabular and morph branches are pre-imputed by FeaturePreprocessor, but
# guard against any NaNs that slip through alignment so GBR/ABR/XGB don't
# crash. Median imputer fit on train only.
if np.isnan(X_train_ext).any() or np.isnan(X_val_ext).any() or np.isnan(X_test_ext).any():
    from sklearn.impute import SimpleImputer
    _imp_ext = SimpleImputer(strategy="median")
    X_train_ext = _imp_ext.fit_transform(X_train_ext)
    X_val_ext   = _imp_ext.transform(X_val_ext)
    X_test_ext  = _imp_ext.transform(X_test_ext)
    print("Imputed residual NaNs (median per feature, fit on train only).")


In [ ]:
# Train Cycle 1+2 model (4 outputs) — splits already done in cell 36.
trainer_ext = ModelTrainer(config, n_estimators=100)

print(f"Cycle 1+2 splits: train={X_train_ext.shape[0]}, val={X_val_ext.shape[0]}, test={X_test_ext.shape[0]}")

print()
print("Training Cycle 1+2 ensemble models...")
print("=" * 60)
best_ext = trainer_ext.train_and_evaluate(
    X_train_ext, Y_train_ext,
    X_val_ext, Y_val_ext,
    target_columns_ext,
    track_learning_curves=False
)

test_results_ext = trainer_ext.evaluate_all_on_test(X_test_ext, Y_test_ext, target_columns_ext)


In [ ]:
# Predicted vs actual for the 4-output Cycle 1+2 model
Y_pred_ext = trainer_ext.best_model.predict(X_test_ext)
if Y_pred_ext.ndim == 1:
    Y_pred_ext = Y_pred_ext.reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, (ax, col) in enumerate(zip(axes, target_columns_ext)):
    y_true = Y_test_ext[:, i]
    y_pred = Y_pred_ext[:, i]

    ax.scatter(y_true, y_pred, alpha=0.6, edgecolors='white', linewidth=0.5, s=60)
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2)
    ax.set_xlabel('Actual', fontsize=10)
    ax.set_ylabel('Predicted', fontsize=10)
    ax.set_title(col, fontsize=10)

    r2 = r2_score(y_true, y_pred)
    ax.annotate(f'R2={r2:.3f}', xy=(0.05, 0.90), xycoords='axes fraction',
                fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Cycle 1+2 Model — {trainer_ext.best_model_name} ({len(df_c1c2)} samples, 4 targets)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison: Cycle 1 only vs Cycle 1+2
print("=" * 60)
print("MODEL COMPARISON: Cycle 1 vs Cycle 1+2")
print("=" * 60)

print(f"\n{'':30s} {'Cycle 1 Only':>15s} {'Cycle 1+2':>15s}")
print(f"{'':30s} {'(82 samples)':>15s} {'(52 samples)':>15s}")
print("-" * 60)
print(f"{'Targets':30s} {'2':>15s} {'4':>15s}")
print(f"{'Best model':30s} {trainer.best_model_name:>15s} {trainer_ext.best_model_name:>15s}")
print(f"{'Test R2 (avg)':30s} {test_results[trainer.best_model_name]['R2']:>15.4f} {test_results_ext[trainer_ext.best_model_name]['R2']:>15.4f}")
print(f"{'Test MAE (avg)':30s} {test_results[trainer.best_model_name]['MAE']:>15.2f} {test_results_ext[trainer_ext.best_model_name]['MAE']:>15.2f}")
print(f"{'Test RMSE (avg)':30s} {test_results[trainer.best_model_name]['RMSE']:>15.2f} {test_results_ext[trainer_ext.best_model_name]['RMSE']:>15.2f}")
print("=" * 60)


## 8. Log Run Metrics

Records all key metrics from this run to `runs/metrics_log.csv` for longitudinal tracking.
Call `m.add_tag()` before `log_run(m)` to annotate what changed (e.g. `smogn=True`, `backbone=resnet50`).

In [ ]:
# ── Metrics logging ──────────────────────────────────────────────────────────
# Appends one row to runs/metrics_log.csv every time this cell executes.
# Edit add_tag() to annotate what changed in this run.

from src.metrics_logger import RunMetrics, log_run, load_log
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ── Cycle 1 metrics ───────────────────────────────────────────────────────────
best = trainer.best_model_name
c1_r2_avg   = test_results[best]["R2"]
c1_mae_avg  = test_results[best]["MAE"]
c1_rmse_avg = test_results[best]["RMSE"]

per_target = {}
for i, col in enumerate(target_columns):
    short = "holdingtemp" if "temp" in col else "holdingtime"
    r2   = r2_score(Y_test[:, i], Y_pred[:, i])
    mae  = mean_absolute_error(Y_test[:, i], Y_pred[:, i])
    rmse = float(np.sqrt(mean_squared_error(Y_test[:, i], Y_pred[:, i])))
    per_target[short] = (r2, mae, rmse)

# ── CV R² (from cv_scores dict) ────────────────────────────────────────────────
try:
    _cv_arr  = cv_scores.get("RF", cv_scores.get("ABR"))
    _cv_mean = float(_cv_arr.mean())
    _cv_std  = float(_cv_arr.std())
except Exception:
    _cv_mean = _cv_std = None

# ── Bayesian-tuned R² ────────────────────────────────────────────────────────
try:
    _bayes_tag  = f"bayes_best={best_tuned_name},r2={best_tuned_r2:.4f}"
    _bayes_delta = bayes_eval[best_tuned_name]["delta"]
    _bayes_tag  += f",delta={_bayes_delta:+.4f}"
    # Override CV R² with the tuned estimate if it beats the untuned
    if best_tuned_r2 > (_cv_mean or 0):
        _cv_mean = best_tuned_r2
        _cv_std  = bayes_eval[best_tuned_name]["tuned_std"]
except Exception:
    _bayes_tag = None

# ── Cycle 1+2 metrics (optional) ─────────────────────────────────────────────
try:
    best_ext      = trainer_ext.best_model_name
    c1c2_r2_avg   = test_results_ext[best_ext]["R2"]
    c1c2_mae_avg  = test_results_ext[best_ext]["MAE"]
    c1c2_rmse_avg = test_results_ext[best_ext]["RMSE"]
except Exception:
    best_ext = c1c2_r2_avg = c1c2_mae_avg = c1c2_rmse_avg = None

# ── Backbone used ─────────────────────────────────────────────────────────────
try:
    _backbone = BACKBONE
except NameError:
    _backbone = "none"

# ── Feature stream dimensions ─────────────────────────────────────────────────
_n_image   = X_img_train.shape[1]   if X_img_train   is not None else None
_n_morph   = X_morph_train.shape[1] if X_morph_train is not None else None
_n_tabular = X_tab_train.shape[1]   if X_tab_train   is not None else None
try:
    _morph_matched = int((~np.isnan(X_morph_all).any(axis=1)).sum())
except Exception:
    _morph_matched = None

# ── Build RunMetrics ──────────────────────────────────────────────────────────
m = RunMetrics(notebook="microstructure_debug")
m.set_dataset(
    n_samples=len(df_c1), n_features=X_train.shape[1], backbone=_backbone,
    n_image_features=_n_image, n_morph_features=_n_morph,
    n_tabular_features=_n_tabular, morph_rows_matched=_morph_matched,
)
m.set_c1(
    best_model=best,
    test_r2_avg=c1_r2_avg, test_mae_avg=c1_mae_avg, test_rmse_avg=c1_rmse_avg,
    per_target=per_target, cv_r2_mean=_cv_mean, cv_r2_std=_cv_std,
)
if best_ext is not None:
    m.set_c1c2(best_model=best_ext,
               test_r2_avg=c1c2_r2_avg, test_mae_avg=c1c2_mae_avg, test_rmse_avg=c1c2_rmse_avg)

# ── Auto-tag from this run ────────────────────────────────────────────────────
_auto_tags = []
if _bayes_tag:
    _auto_tags.append(_bayes_tag)
if _backbone != "none":
    _auto_tags.append(f"backbone={_backbone}")
if X_morph_train is not None:
    _auto_tags.append("morph=True")
m.add_tag(*_auto_tags)
# Add your own annotations here, e.g.:  m.add_tag("smogn=True", "pruned=True")

# ── Write + display ───────────────────────────────────────────────────────────
log_path = log_run(m)
m.print_summary()
print(f"\nAppended to: {log_path}")

# Show full history when more than one run exists
df_log = load_log()
if len(df_log) > 1:
    print(f"\nAll {len(df_log)} runs logged:")
    view_cols = ["timestamp", "git_commit", "backbone",
                 "c1_best_model", "c1_test_r2_avg",
                 "c1_holdingtemp_r2", "c1_holdingtime_r2",
                 "c1_cv_r2_mean", "c1c2_test_r2_avg", "tags"]
    print(df_log[[c for c in view_cols if c in df_log.columns]].to_string(index=False))
# ── RunStore: write manifest + history + update visualisation ─────────────────
_run_row = m.to_row()
_store.write_manifest({
    'backbone':           _backbone,
    'n_samples':          len(df_c1),
    'n_features':         int(X_train.shape[1]),
    'n_image_features':   _n_image,
    'n_morph_features':   _n_morph,
    'n_tabular_features': _n_tabular,
    'c1_best_model':      best,
    'c1_test_r2_avg':     c1_r2_avg,
    'c1_holdingtemp_r2':  per_target.get('holdingtemp', (None,))[0],
    'c1_holdingtime_r2':  per_target.get('holdingtime', (None,))[0],
    'c1_cv_r2_mean':      _cv_mean,
    'c1_cv_r2_std':       _cv_std,
    'c1c2_best_model':    best_ext,
    'c1c2_test_r2_avg':   c1c2_r2_avg,
    'tags':               '; '.join(_auto_tags),
})

_store.append_history({
    'backbone':           _backbone,
    'n_samples':          len(df_c1),
    'n_features':         int(X_train.shape[1]),
    'best_model':         best,
    'c1_test_r2_avg':     c1_r2_avg,
    'c1_holdingtemp_r2':  per_target.get('holdingtemp', (None,))[0],
    'c1_holdingtime_r2':  per_target.get('holdingtime', (None,))[0],
    'c1_cv_r2_mean':      _cv_mean,
    'c1_cv_r2_std':       _cv_std,
    'c1c2_test_r2_avg':   c1c2_r2_avg,
    'tags':               '; '.join(_auto_tags),
})

# Copy model comparison artifact if it was saved
try:
    _mc_src = _run_dir / 'model_comparison.png'
    if not _mc_src.exists():
        import glob as _glob
        _candidates = _glob.glob('../runs/*/model_comparison.png')
        # already in run_dir from cell 35 savefig
    pass
except Exception:
    pass

# Regenerate run-family history visualisation
from pathlib import Path as _Path
metrics_viz.plot_history('debug', save=True, show=False)
print(f'History visualisation -> runs/debug/metrics_history.png')
print(f'Run artifacts stored  -> {_run_dir}')
